# SoundStream &mdash; Demo

This notebook demonstrates an example usage of SoundStream. It downloads the audio, given by url, encodes it to codebook indices and decodes it back.

**Usage**: set `AUDIO_URL` in `Downloading audio` section, and then run all cells.

### Setting up environment and loading model

In [ ]:
# clonning repository and setting up environment 
!git clone https://github.com/Alekseyka20x/SoundStream
%cd SoundStream
!pip install -q -r requirements.txt

In [ ]:
from src.models.generator import SoundStream
import gdown

# downloading model weights and config from google drive
model_file_id = "1VdAG_X5E2tTEqI7jjmpvM_p7iZas7WET"
save_path = "best_model.pth"
gdown.download(id=model_file_id, output=save_path)

# loading model from disk
model = SoundStream.from_pretrained(save_path)

### Downloading audio

Set `AUDIO_URL` in the cell below.

In [ ]:
# replace this url with your audio url
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

In [ ]:
import wget
import torchaudio
import os

# downloading audio to disk
file_name = "audio.wav"
if os.path.exists(file_name):
    os.remove(file_name)

wget.download(AUDIO_URL, file_name)
audio, sample_rate = torchaudio.load(file_name)
print(f"Sample audio was loaded. Shape: {list(audio.shape)}; Sample rate: {sample_rate}")

In [ ]:
# only mono audio is supported
if audio.dim() == 1:
    audio = audio.unsqueeze(0)
if audio.shape[0] != 1:
    print("Converting to mono")
    audio = audio.sum(dim=0, keepdim=True)

### Encoding audio and decoding it back. Displaying results

In [ ]:
# encoding audio
encoded_audio = model.encode(audio, sample_rate=sample_rate)

# decoding audio
decoded_audio = model.decode(encoded_audio, sample_rate=sample_rate)

In [ ]:
# displaying results
from IPython.display import Audio

print("Original:")
display(Audio(audio, rate=sample_rate))

print("\nReconstructed:")
display(Audio(decoded_audio, rate=sample_rate))